# 02 Feature Engineering

This notebook creates the analytical layer for the Q1 2025 NYC Yellow Taxi BI project.

Preprocessing/data cleaning is treated as already completed. The notebook therefore does not rewrite the preprocessing decisions; it loads the cleaned dataset when one is available, adds business-analysis features, joins taxi-zone metadata, validates derived values, and saves the analysis-ready output used by the EDA and dashboard notebooks.

## TOR Alignment

The TOR requires an analytical dataset or curated analytical outputs that support:

- **Univariate EDA:** trip volume over time; distributions of trip distance, trip duration, fare amount, total amount, passenger count; frequencies of payment type and rate code.
- **Bivariate EDA:** hour of day versus trip volume; geography versus revenue or trip count; payment type versus tip or total amount; distance versus total amount; weekday versus demand.
- **Multivariate EDA:** time x geography x demand; geography x payment type x revenue; distance x duration x total amount; month x hour x borough/zone; pre/post January 5, 2025 patterns where `cbd_congestion_fee` is relevant.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
TAXI_DATA_PATH = Path("Data/project_master_clean.parquet")
ZONE_LOOKUP_PATH = Path("Data/taxi_zone_lookup.csv")
OUTPUT_DIR = Path("Data/processed/analytical")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYTICAL_OUTPUT_PATH = OUTPUT_DIR / "analytical.parquet"
ANALYTICAL_SAMPLE_OUTPUT_PATH = OUTPUT_DIR / "yellow_taxi_q1_2025_analytical_sample.csv"

## Load Cleaned Trip Data

The notebook uses the completed cleaned master file directly: `Data/project_master_clean.parquet`. This is the only taxi trip input for feature engineering.

In [ ]:
taxi = pd.read_parquet(TAXI_DATA_PATH)

print(f"Loaded taxi data from: {TAXI_DATA_PATH}")
print(f"Rows: {taxi.shape[0]:,} | Columns: {taxi.shape[1]:,}")
taxi.head()

## Load Taxi Zone Lookup Metadata

The analytical layer uses `Data/taxi_zone_lookup.csv` to attach borough, zone, and service-zone metadata for both pickup and dropoff locations.

In [ ]:
zones = pd.read_csv(ZONE_LOOKUP_PATH)
zones = zones.rename(columns={"LocationID": "location_id", "Borough": "borough", "Zone": "zone"})

required = ["location_id", "borough", "zone", "service_zone"]
missing = [column for column in required if column not in zones.columns]
if missing:
    raise ValueError(f"Zone lookup is missing required columns: {missing}")

zones = zones[required].drop_duplicates("location_id")
zones["location_id"] = pd.to_numeric(zones["location_id"], errors="coerce").astype("Int64")
zones["service_zone"] = zones["service_zone"].fillna("Unknown")

print(f"Zone lookup rows: {zones.shape[0]:,}")
zones.head()

## Engineer Business-Analysis Features

Derived values are created with explicit validation. Impossible or invalid engineered results are set to missing values and tracked with boolean flags instead of silently contaminating later EDA.

In [ ]:
MAX_VALID_DURATION_MIN = 24 * 60
MAX_VALID_TIP_PERCENT = 100
MAX_VALID_SPEED_MPH = 100


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pickup_ts = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    dropoff_ts = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    df["pickup_hour"] = pickup_ts.dt.hour.astype("Int64")
    df["pickup_day"] = pickup_ts.dt.day_name()
    df["pickup_month"] = pickup_ts.dt.to_period("M").astype("string")
    df["pickup_date"] = pickup_ts.dt.date
    df["is_weekend"] = pickup_ts.dt.dayofweek.ge(5).astype("boolean")

    duration_min = (dropoff_ts - pickup_ts).dt.total_seconds() / 60
    df["invalid_trip_duration"] = duration_min.isna() | duration_min.le(0) | duration_min.gt(MAX_VALID_DURATION_MIN)
    df["trip_duration_min"] = duration_min.mask(df["invalid_trip_duration"])
    return df


def add_financial_and_speed_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    fare = pd.to_numeric(df["fare_amount"], errors="coerce")
    tip = pd.to_numeric(df["tip_amount"], errors="coerce")
    distance = pd.to_numeric(df["trip_distance"], errors="coerce")
    duration = pd.to_numeric(df["trip_duration_min"], errors="coerce")

    tip_percent = (tip / fare) * 100
    df["invalid_tip_percent"] = (
        tip_percent.isna()
        | fare.le(0)
        | tip.lt(0)
        | tip_percent.lt(0)
        | tip_percent.gt(MAX_VALID_TIP_PERCENT)
    )
    df["tip_percent"] = tip_percent.mask(df["invalid_tip_percent"])

    avg_speed = distance / (duration / 60)
    df["invalid_avg_speed"] = (
        avg_speed.isna()
        | distance.le(0)
        | duration.le(0)
        | avg_speed.lt(0)
        | avg_speed.gt(MAX_VALID_SPEED_MPH)
    )
    df["avg_speed_mph"] = avg_speed.mask(df["invalid_avg_speed"])
    return df


def add_cbd_period_feature(df: pd.DataFrame) -> pd.DataFrame:
    """Support the TOR's pre/post January 5, 2025 CBD congestion fee analysis."""
    df = df.copy()
    pickup_ts = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["cbd_period"] = np.select(
        [pickup_ts.lt(pd.Timestamp("2025-01-05")), pickup_ts.ge(pd.Timestamp("2025-01-05"))],
        ["pre_2025_01_05", "post_2025_01_05"],
        default=pd.NA,
    )
    return df


taxi_features = (
    taxi.pipe(add_time_features)
        .pipe(add_financial_and_speed_features)
        .pipe(add_cbd_period_feature)
)

feature_columns = [
    "pickup_hour", "pickup_day", "pickup_month", "pickup_date", "is_weekend",
    "trip_duration_min", "tip_percent", "avg_speed_mph", "cbd_period",
    "invalid_trip_duration", "invalid_tip_percent", "invalid_avg_speed",
]

taxi_features[feature_columns].head()

## Merge Pickup and Dropoff Zone Metadata

Pickup and dropoff joins are kept separate and renamed clearly so later EDA can group by either origin or destination geography without ambiguity.

In [ ]:
def merge_zone_metadata(df: pd.DataFrame, zones: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    pickup_zones = zones.rename(
        columns={
            "location_id": "PULocationID",
            "borough": "pickup_borough",
            "zone": "pickup_zone",
            "service_zone": "pickup_service_zone",
        }
    )
    dropoff_zones = zones.rename(
        columns={
            "location_id": "DOLocationID",
            "borough": "dropoff_borough",
            "zone": "dropoff_zone",
            "service_zone": "dropoff_service_zone",
        }
    )

    df = df.merge(pickup_zones, on="PULocationID", how="left", validate="many_to_one")
    df = df.merge(dropoff_zones, on="DOLocationID", how="left", validate="many_to_one")

    for column in ["pickup_borough", "pickup_zone", "pickup_service_zone", "dropoff_borough", "dropoff_zone", "dropoff_service_zone"]:
        df[column] = df[column].fillna("Unknown")

    return df


analytical = merge_zone_metadata(taxi_features, zones)

zone_columns = [
    "PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone",
]
analytical[zone_columns].head()

## Analytical Layer Quality Checks

These checks verify that the engineered columns exist, the zone metadata has been attached for both trip ends, and invalid engineered values are controlled before saving.

In [ ]:
required_output_columns = [
    "pickup_hour", "pickup_day", "pickup_month", "pickup_date", "is_weekend",
    "trip_duration_min", "tip_percent", "avg_speed_mph",
    "pickup_borough", "pickup_zone", "pickup_service_zone",
    "dropoff_borough", "dropoff_zone", "dropoff_service_zone",
]
missing_output_columns = [column for column in required_output_columns if column not in analytical.columns]
if missing_output_columns:
    raise AssertionError(f"Missing analytical columns: {missing_output_columns}")

quality_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "invalid_trip_duration_rows",
        "invalid_tip_percent_rows",
        "invalid_avg_speed_rows",
        "unknown_pickup_zone_rows",
        "unknown_dropoff_zone_rows",
    ],
    "value": [
        len(analytical),
        analytical.shape[1],
        int(analytical["invalid_trip_duration"].sum()),
        int(analytical["invalid_tip_percent"].sum()),
        int(analytical["invalid_avg_speed"].sum()),
        int(analytical["pickup_zone"].eq("Unknown").sum()),
        int(analytical["dropoff_zone"].eq("Unknown").sum()),
    ],
})
quality_summary

## Save Analytical Output

The parquet file is the main analytical dataset for the downstream EDA, reporting, and dashboard notebooks. A small CSV sample is also written for quick inspection without loading the full dataset.

In [ ]:
analytical.to_parquet(ANALYTICAL_OUTPUT_PATH, index=False)
analytical.head(10_000).to_csv(ANALYTICAL_SAMPLE_OUTPUT_PATH, index=False)

print(f"Saved analytical parquet: {ANALYTICAL_OUTPUT_PATH}")
print(f"Saved inspection sample: {ANALYTICAL_SAMPLE_OUTPUT_PATH}")
print(f"Final shape: {analytical.shape[0]:,} rows x {analytical.shape[1]:,} columns")

## Feature-to-EDA Mapping

| Feature | Why it matters | TOR coverage enabled |
|---|---|---|
| `pickup_hour` | Identifies intraday demand peaks and operating pressure. | Bivariate: hour vs trip volume; Multivariate: month x hour x borough/zone. |
| `pickup_day` | Separates weekday patterns and named-day effects. | Bivariate: weekday vs demand. |
| `pickup_month` | Supports January-February-March comparisons. | Univariate: trip volume over time; Multivariate: month x hour x geography. |
| `pickup_date` | Enables daily trend lines and pre/post policy timing. | Univariate: trip volume over time; Multivariate: pre/post January 5 CBD analysis. |
| `is_weekend` | Compares commuter-weekday demand with leisure-weekend demand. | Bivariate: weekday/weekend vs demand; Multivariate: time x geography x demand. |
| `trip_duration_min` | Required for duration distributions and speed/distance diagnostics. | Univariate: trip duration; Multivariate: distance x duration x total amount. |
| `tip_percent` | Normalizes tipping behavior across fare sizes. | Bivariate: payment type vs tip; Multivariate: geography x payment type x revenue. |
| `avg_speed_mph` | Flags operational congestion and supports distance-duration consistency checks. | Multivariate: distance x duration x total amount; geography/time diagnostics. |
| Pickup/dropoff borough, zone, service zone | Converts location IDs into business-readable geography. | Bivariate: geography vs revenue/trip count; Multivariate: time x geography x demand. |
| `cbd_period` | Creates a clean pre/post label for congestion-pricing analysis. | Multivariate: pre/post January 5, 2025 patterns where `cbd_congestion_fee` is relevant. |